In [7]:
import pyterrier as pt
from pyterrier_t5 import MonoT5ReRanker

from code_base.wiki_loader_pyt import wiki_loader_pyt

# Constants
DATA_PATH = "/mnt/c/Users/prath/Documents/Uni(Tudelft)/MSc/MasterThesis/data/2wikimultihopQA"
SPLIT = "test"

# Load dataframes
docs_df, topics_df, qrels_df = wiki_loader_pyt(DATA_PATH, SPLIT)

docs_df = docs_df.dropna(subset=['docno', 'text'])
docs_df['docno'] = docs_df['docno'].astype(str)
docs_df = docs_df[docs_df['text'].str.strip() != '']
docs_df.info()

Processing corpus...


100%|██████████| 563424/563424 [00:00<00:00, 1021826.29it/s]


Created docs_df with 495034 documents.

Processing test set for topics and qrels...


100%|██████████| 12576/12576 [00:00<00:00, 180062.08it/s]


Created topics_df with 12576 queries.
Created qrels_df with 125760 relevance judgments.
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 495034 entries, 0 to 495033
Data columns (total 2 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   docno   495034 non-null  object
 1   text    495034 non-null  object
dtypes: object(2)
memory usage: 7.6+ MB


In [8]:
# Uncomment if you want to reindex
indexer = pt.IterDictIndexer("./index/wiki_musique_index", overwrite=True, meta={'docno': 256, 'text': 4096})
index_ref = indexer.index(docs_df.to_dict(orient='records'))

# index_path = "./index/wiki_musique_index/data.properties"  # path to the Terrier index metadata
# index_ref = pt.IndexFactory.of(index_path)


print(index_ref)

16:47:55.548 [ForkJoinPool-1-worker-1] WARN org.terrier.structures.indexing.Indexer -- Adding an empty document to the index (Timeline of the Insurgency in Northeast India) - further warnings are suppressed
16:52:08.238 [ForkJoinPool-1-worker-1] WARN org.terrier.structures.indexing.Indexer -- Indexed 2 empty documents
<org.terrier.querying.IndexRef at 0x7e28ebd7e450 jclass=org/terrier/querying/IndexRef jself=<LocalRef obj=0x48c10440 at 0x7e2903a04e30>>


In [9]:
def preprocess_query(query):
    # remove punctuation and special characters
    query = ''.join(e for e in query if e.isalnum() or e.isspace())
    # convert to lowercase
    query = query.lower()
    return query


topics_df['query'] = topics_df['query'].apply(preprocess_query)

In [ ]:
#test to see if index is built correctly

t_limit = 100
c = 10
bm25 = pt.terrier.Retriever(index_ref, wmodel="BM25", controls={})

pipline = bm25 % c >> pt.text.get_text(index_ref, "text") >> MonoT5ReRanker(verbose=False, batch_size=16)

experiment = pt.Experiment(
    [bm25 % c, pipline],
    topics_df[:t_limit],
    qrels_df,
    batch_size=10,
    eval_metrics=["map", NDCG @ 10, R @ 5],
    names=["BM25", "BM25+MonoT5"],
    verbose=True,
    save_dir=DATA_PATH,
)


In [ ]:
experiment

In [ ]:
# Step 1: Setup up dataset AND constant values [DONE for wiki, need to do for ambig]
# Step 2: Convert Query to n-Query [Maybe, need more testing issues - how to convert llm reposon to n-query?]
# Step 3: Create a function to combine n-query doc ranking to k-doc ranking [DONE]
# Step 4: Setup the rag pipeline

In [2]:
import pyterrier as pt

bm25_wiki = pt.terrier.Retriever.from_dataset('irds:wikiclir', 'terrier_stemmed', wmodel='BM25')

Java started (triggered by Retriever.from_dataset) and loaded: pyterrier.java, pyterrier.terrier.java [version=5.11 (build: craig.macdonald 2025-01-13 21:29), helper_version=0.0.8]


NotImplementedError: IRDSDataset('wikiclir') does not support get_index

'brown'

retrival dataset - Bright dataset

